## Setup

Runs on Colab, no-op locally. Run it first, before anything else.

`torch` and `numpy` are deliberately left alone: Colab's builds are CUDA-matched, and replacing them costs minutes and forces a runtime restart.

In [ ]:
# --- Colab setup ---------------------------------------------------
# Installs only what Colab is missing. Locally this whole cell is skipped.
import subprocess, sys

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "transformers==5.7.0",
            "datasets>=5.0.1",
        ],
        check=True,
    )
    print("Colab: dependencies installed.")
else:
    print("Local environment: nothing to install.")


# `notebooks/utils.py` is not on Colab's path, so `get_device` is defined here.
import torch


def get_device() -> torch.device:
    """Return the best available device for PyTorch operations."""
    if torch.cuda.is_available():
        print("Using GPU for PyTorch operations.")
        return torch.device("cuda")
    elif torch.backends.mps.is_available():
        print("Using Apple MPS for PyTorch operations.")
        return torch.device("mps")
    else:
        print("Using CPU for PyTorch operations.")
        return torch.device("cpu")


# A3 Does BERT capture truthfulness of statements?

So far we have only *looked* at representations. Now we use them: a **prediction
task**.

The dataset is the [Trilemma of Truth](https://huggingface.co/datasets/carlomarxx/trilemma-of-truth)
collection of factual statements, each labelled `true`, `false`, or `neither`.
1. We drop `neither` and keep a clean binary problem. That is an oversimplification, but it
   makes a good example.
2. We work only with the `city_locations` part of the dataset.

> Given a statement, is it true?

Nothing is trained. We freeze `ModernBERT`, take the representation it already has,
and fit a **logistic regression** on top: the same recipe we use on life
sequences later. The question is whether the truth of a sentence is *separable* from the model's internal states, and if so, **at which layer**.

In [ ]:
import numpy as np
import pandas as pd
import torch
from matplotlib import pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from transformers import AutoTokenizer, AutoModelForMaskedLM

model_id = "answerdotai/ModernBERT-base"

device = get_device()
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForMaskedLM.from_pretrained(model_id).eval().to(device)

N_LAYERS = model.config.num_hidden_layers + 1  # +1 for the embedding output


## 1. The data

In [ ]:
from datasets import DatasetDict, load_dataset

ds = load_dataset("carlomarxx/trilemma-of-truth", "city_locations")


def preprocess_data(ds: DatasetDict, split: str = "train") -> pd.DataFrame:
    """One split of the dataset as a frame, keeping the columns we use.

    `is_neither` rows are dropped, which leaves a two-class task with `correct` as the label.

    Args:
        ds: The loaded dataset, with one entry per split.
        split: Which split to return, one of train, test or validation.
    Returns:
        One row per statement.
    """
    df = (
        ds[split]
        .to_pandas()
        .query("is_neither == False")
        .loc[
            :,
            [
                "statement",
                "object_1",
                "object_2",
                "correct",
                "negation",
            ],
        ]
    )

    return df


df_train = preprocess_data(ds, "train")
df_test = preprocess_data(ds, "test")
df_val = preprocess_data(ds, "validation")
df_train.head()


#### Dataset Summary

In [ ]:
def summarise(d: pd.DataFrame) -> pd.Series:
    """Per-split counts: label balance, negation rate, entity coverage."""
    return pd.Series(
        {
            "statements": len(d),
            "true": int(d.correct.sum()),
            "false": int((~d.correct).sum()),
            "negated": int(d.negation.sum()),
            "unique cities": d.object_1.nunique(),
            "unique countries": d.object_2.nunique(),
        }
    )


summary = pd.DataFrame(
    {
        "train": summarise(df_train),
        "test": summarise(df_test),
        "validation": summarise(df_val),
    }
)
summary

#### Your Turn

In [ ]:
# Display a few negated and affirmative statements from the training set

# Display a few true and false  statements from the training set

## 2. Baselines first

Never look at a model score before you know what "unimpressive" looks like. Three
baselines:

1. **Chance**: the positive rate, which is what a random ranker scores under average
   precision.
2. **The negation flag alone**: one binary feature, no text.
3. **TF-IDF + logistic regression**: word counts. It sees the actual words, just not
   their *relationship*. Every statement here comes from the same template, so word
   counts carry almost no signal and this lands at or below the chance line. That is a
   result about the dataset, not a flaw in the baseline.


We use [Average Precision](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.average_precision_score.html#sklearn.metrics.average_precision_score) to evaluate the performance.

<details>
<summary><b>How is the data split?</b></summary>
We use the train and test splits that ship with the dataset. They are <b>not</b>
entity-disjoint: 653 of the 763 test cities also appear in train, so the probe can
memorise part of the answer at the city level. Grouping the split on `object_1`
instead would force it to generalise to cities it has never seen.
</details>


In [ ]:
from sklearn.metrics import average_precision_score


def baselines(tr: pd.DataFrame, te: pd.DataFrame) -> pd.Series:
    """Average precision, to match the probe.

    A random ranker scores the positive rate, so that, rather than 0.5, is the floor
    worth beating.
    """
    y_tr, y_te = tr.correct.values.astype(int), te.correct.values.astype(int)
    scores = {"chance (positive rate)": y_te.mean()}

    scores["negation flag alone"] = average_precision_score(
        y_te,
        LogisticRegression()
        .fit(tr.negation.values.reshape(-1, 1), y_tr)
        .predict_proba(te.negation.values.reshape(-1, 1))[:, 1],
    )

    tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2)
    scores["TF-IDF + logistic reg."] = average_precision_score(
        y_te,
        LogisticRegression(max_iter=2000)
        .fit(tfidf.fit_transform(tr.statement), y_tr)
        .predict_proba(tfidf.transform(te.statement))[:, 1],
    )

    return pd.Series(scores, name="average_precision")


base_cities = baselines(df_train, df_test)
base_cities.round(3)

**Note**: Usually, you would also estimate the confidence intervals for the metric that you use (but this is out of scope)

## 3. One vector per statement, at every layer

`output_hidden_states=True` gives us all the representations over the 23 layers. 
For each layer we **mean-pool** over the real tokens, averaging with the attention mask so
padding does not leak in.

In [ ]:
def embed_statements(
    statements: list[str], layer: int | None = None, batch_size: int = 32
) -> np.ndarray:
    """Mean-pooled sentence vector, averaging over the real tokens only.

    layer=None -> every layer, shape (n, n_layers, hidden)
    layer=k    -> just layer k,  shape (n, hidden)
    """
    layers = list(range(N_LAYERS)) if layer is None else [layer]
    X = np.zeros(
        (len(statements), len(layers), model.config.hidden_size), dtype=np.float32
    )

    for i in range(0, len(statements), batch_size):
        batch = statements[i : i + batch_size]
        enc = tokenizer(batch, return_tensors="pt", padding=True, truncation=True).to(
            device
        )
        with torch.no_grad():
            out = model(**enc, output_hidden_states=True)
        mask = enc["attention_mask"].unsqueeze(-1).float()  # (b, seq, 1)
        for j, l in enumerate(layers):
            h = out.hidden_states[l]
            X[i : i + batch_size, j] = ((h * mask).sum(1) / mask.sum(1)).cpu().numpy()
        print(f"  {min(i + batch_size, len(statements))}/{len(statements)}", end="\r")

    return X[:, 0] if layer is not None else X


LAYER = 17
v = embed_statements(df_train.statement.head(3).tolist(), layer=LAYER)
print(f"layer {LAYER}: {v.shape[1]} numbers per statement")
for statement, vec in zip(df_train.statement.head(3), v):
    print(f"  {statement[:50]:<52} -> [{vec[0]:+.3f}, {vec[1]:+.3f}, ...]")


## 4. A probe per layer

One logistic regression per layer, all on the same split. The probe is
deliberately weak: if a *linear* model can read the label off the
representation, the information is genuinely there and not something the probe
invented.

#### Probe for single layer

In [ ]:
def probe(
    X_tr: np.ndarray, X_te: np.ndarray, tr: pd.DataFrame, te: pd.DataFrame
) -> float:
    """Average precision of a logistic regression fitted on one layer's vectors.

    Args:
        X_tr: Train statement vectors, shape (n_train, hidden).
        X_te: Test statement vectors, shape (n_test, hidden).
        tr: Train frame, used for the `correct` label.
        te: Test frame, used for the `correct` label.
    Returns:
        Average precision on the test statements.
    """
    y_tr, y_te = tr.correct.values.astype(int), te.correct.values.astype(int)
    p = LogisticRegression(max_iter=4000).fit(X_tr, y_tr).predict_proba(X_te)[:, 1]
    return average_precision_score(y_te, p)


X_train = embed_statements(df_train.statement.tolist(), layer=LAYER)
X_test = embed_statements(df_test.statement.tolist(), layer=LAYER)
print(f"\nshapes {X_train.shape} / {X_test.shape}  = (statements, hidden)")


ap_cities = probe(X_train, X_test, df_train, df_test)

print(f"baselines:\n{base_cities.round(3).to_string()}")
print(f"\nTransformer\n\tlayer {LAYER}: {ap_cities:.3f}")

#### Probe across layers

In [ ]:
INK, MUTED, GRID = "#56B4E9", "#009E73", "#DCDCD8"
LAYERS = [0, 4, 8, 12, 16, 20, 22]

# Embed all statements for all layers
X_train_all = embed_statements(df_train.statement.tolist())
X_test_all = embed_statements(df_test.statement.tolist())

acc_cities = pd.Series(
    {
        layer: probe(X_train_all[:, layer], X_test_all[:, layer], df_train, df_test)
        for layer in LAYERS
    },
    name="average_precision",
).rename_axis("layer")


def plot_layers(acc: pd.Series, base: pd.Series, title: str) -> plt.Axes:
    """Average precision per layer, drawn against the baselines as reference lines.

    Args:
        acc: Average precision indexed by layer.
        base: Baseline scores, drawn as labelled horizontal lines.
        title: Figure title.
    Returns:
        The axes, so the caller can annotate further.
    """
    fig, ax = plt.subplots(figsize=(9, 5), dpi=120)

    lo = min(acc.min(), base.min()) - 0.015
    hi = acc.max() + 0.05  # headroom so the peak label clears the title
    ax.set_ylim(lo, hi)

    # one data series -> one colour, no legend box; the title names it
    ax.plot(
        acc.index,
        acc.values,
        color=INK,
        lw=2,
        marker="o",
        ms=5,
        zorder=3,
        clip_on=False,
    )

    # Baselines are reference lines, not series: labelled directly just outside the
    # axes, and nudged apart when two of them land close together.
    gap = (hi - lo) * 0.05
    label_y = -np.inf
    for name, value in base.sort_values().items():
        ax.axhline(value, color=MUTED, lw=1.2, ls="--", zorder=2)
        label_y = max(value, label_y + gap)
        ax.annotate(
            f"{name} {value:.2f}",
            (1.01, label_y),
            xycoords=("axes fraction", "data"),
            va="center",
            ha="left",
            fontsize=8,
            color=MUTED,
        )

    peak = int(acc.idxmax())
    ax.annotate(
        f"layer {peak}: {acc.max():.3f}",
        (peak, acc.max()),
        textcoords="offset points",
        xytext=(0, 10),
        ha="center",
        fontsize=9,
        color=INK,
        fontweight="bold",
    )

    ax.set_xlabel("layer")
    ax.set_ylabel("average precision, test statements")
    ax.set_title(title, loc="left", fontsize=11, pad=16)
    ax.set_xticks(acc.index)  # the layers actually probed, not evenly spaced positions
    ax.grid(axis="y", color=GRID, lw=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    fig.subplots_adjust(right=0.70)
    return ax


plot_layers(
    acc_cities,
    base_cities,
    "Is this statement true? Linear probe on frozen ModernBERT",
)
plt.show()

### What we appear to have found

The curve climbs out of the baselines and peaks in the upper-middle of the network.
The obvious conclusion: **truth is linearly encoded, and it gets sharper with depth.**

**Important**: We simplified the methodology quite a bit. If you are interested in more up-to-date methods, see the [Trilemma of Truth](https://arxiv.org/abs/2506.23921) paper.
